# Test PYNQ dla IP Leading Edge AXI-Lite

        Notebook sluzy do uruchomienia projektu `Leading_edge_AXI` na platformie PYNQ/ZCU106 oraz do sprawdzenia rejestrow AXI4-Lite rdzenia `leading_edge_ip_lite`.

        Sekwencja testu:
        1. rozpakowanie XSA i kontrola obecnosci plikow `.bit` oraz `.hwh`,
        2. zaladowanie overlay z pliku `.xsa`,
        3. wykrycie instancji IP w `overlay.ip_dict`,
        4. test zapisu/odczytu rejestrow,
        5. uruchomienie pojedynczych przypadkow dla metod LIN, EXP i LOG.


## 0. Przygotowanie uruchomienia

            Przed konfiguracja PL nalezy upewnic sie, ze plik `leading_edge_axi_bd_wrapper.xsa` znajduje sie w katalogu notebooka. Po nieudanej probie konfiguracji zalecany jest restart kernela oraz ponowne wykonanie komorek od poczatku.

            Notebook laduje projekt z pliku XSA, poniewaz ten format zawiera metadane wymagane przez PYNQ.


## 1. Importy i overlay

XSA jest archiwum. W tej wersji PYNQ ladujemy **bezposrednio z `.xsa`** i nie robimy fallbacku na `.bit`, bo plik XSA zawiera metadane wymagane do poprawnej inicjalizacji overlay.

Komorka nadal wyciaga pomocniczo `.bit` i `.hwh`, zeby latwo bylo sprawdzic, co jest w XSA:

- `leading_edge_axi_bd_wrapper.bit`
- `leading_edge_axi_bd_wrapper.hwh`


In [ ]:
from pathlib import Path
import time
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    from pynq import Overlay, MMIO
except ImportError as exc:
    raise ImportError(
        "Ten notebook trzeba uruchomic na obrazie PYNQ, gdzie dostepny jest pakiet pynq."
    ) from exc

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
np.set_printoptions(precision=6, suppress=True)

XSA_FILE = Path("leading_edge_axi_bd_wrapper.xsa")
BIT_FILE = XSA_FILE.with_suffix(".bit")
HWH_FILE = XSA_FILE.with_suffix(".hwh")

if not XSA_FILE.exists():
    raise FileNotFoundError(f"Brak pliku {XSA_FILE}. Wgraj XSA do katalogu notebooka.")

with zipfile.ZipFile(XSA_FILE, "r") as z:
    bit_members = [name for name in z.namelist() if name.endswith(".bit")]
    hwh_members = [name for name in z.namelist() if name.endswith(".hwh")]
    if not bit_members:
        raise FileNotFoundError("W XSA nie znaleziono pliku .bit")
    if not hwh_members:
        raise FileNotFoundError("W XSA nie znaleziono pliku .hwh")

    BIT_FILE.write_bytes(z.read(bit_members[0]))
    HWH_FILE.write_bytes(z.read(hwh_members[0]))

print(f"XSA: {XSA_FILE} ({XSA_FILE.stat().st_size} bajtow)")
print(f"BIT: {BIT_FILE} ({BIT_FILE.stat().st_size} bajtow)")
print(f"HWH: {HWH_FILE} ({HWH_FILE.stat().st_size} bajtow)")

# Programowanie PL: tylko XSA, bez fallbacku na .bit.
# Konfiguracja PL jest wykonywana w jednym kontrolowanym kroku.
ol = Overlay(str(XSA_FILE))
print(f"Overlay zaladowany z XSA: {XSA_FILE}")

print("IP w overlay:")
for name, info in ol.ip_dict.items():
    base = info.get("phys_addr", info.get("base_addr", 0))
    rng = info.get("addr_range", 0)
    print(f"  - {name:35s} base=0x{int(base):08X} range=0x{int(rng):X}")


## 2. Wybór IP

W Block Design instancja powinna nazywac sie `leading_edge_ip_lite_0`.
Jesli Vivado zmieni nazwe, notebook szuka pierwszego IP z `leading_edge` w nazwie.


In [ ]:
IP_NAME = "leading_edge_ip_lite_0"

if IP_NAME in ol.ip_dict:
    ip_info = ol.ip_dict[IP_NAME]
    try:
        ip = getattr(ol, IP_NAME)
        print(f"Wybrano IP jako atrybut overlay: {IP_NAME}")
    except AttributeError:
        ip = MMIO(ip_info["phys_addr"], ip_info["addr_range"])
        print(f"Wybrano IP przez MMIO: {IP_NAME}")
else:
    candidates = [
        name for name in ol.ip_dict
        if "leading_edge" in name.lower() or "leading" in name.lower()
    ]
    if not candidates:
        raise RuntimeError("Nie znaleziono leading_edge_ip_lite w overlay.ip_dict")
    IP_NAME = candidates[0]
    ip_info = ol.ip_dict[IP_NAME]
    ip = MMIO(ip_info["phys_addr"], ip_info["addr_range"])
    print(f"Wybrano IP przez wyszukiwanie: {IP_NAME}")

IP_BASE = int(ip_info.get("phys_addr", ip_info.get("base_addr", 0)))
IP_RANGE = int(ip_info.get("addr_range", 0x1000))
print(f"Adres bazowy: 0x{IP_BASE:08X}, zakres: 0x{IP_RANGE:X}")


## 3. Mapa rejestrow

Aktualna mapa `leading_edge_ip_lite`:

| Offset | Rejestr | Opis |
|---:|---|---|
| `0x00` | CONTROL | `[0]` start pulse, `[2:1]` mode, `[8]` clear_done |
| `0x04` | STATUS | `[0]` busy, `[1]` done, `[2]` overflow, `[3]` core_valid |
| `0x08` | THRESHOLD | prog Q16.16 |
| `0x0C` | T1 | czas probki 1 Q16.16 |
| `0x10` | A1 | amplituda probki 1 Q16.16 |
| `0x14` | T2 | czas probki 2 Q16.16 |
| `0x18` | A2 | amplituda probki 2 Q16.16 |
| `0x1C` | T3 | czas probki 3 Q16.16 |
| `0x20` | A3 | amplituda probki 3 Q16.16 |
| `0x24` | T0_EST | wynik Q16.16 |
| `0x28` | AMAX_EST | wynik amplitudy Q16.16 |
| `0x2C` | VERSION | `0x00010000` |


In [ ]:
REG_CONTROL   = 0x00
REG_STATUS    = 0x04
REG_THRESHOLD = 0x08
REG_T1        = 0x0C
REG_A1        = 0x10
REG_T2        = 0x14
REG_A2        = 0x18
REG_T3        = 0x1C
REG_A3        = 0x20
REG_T0_EST    = 0x24
REG_AMAX_EST  = 0x28
REG_VERSION   = 0x2C

MODE_LINEAR = 0
MODE_EXP    = 1
MODE_LOG    = 2
MODE_NAMES = {
    MODE_LINEAR: "LINEAR",
    MODE_EXP: "EXP",
    MODE_LOG: "LOG",
}

Q16_SCALE = 1 << 16


def to_q16(value: float) -> int:
    """Float -> signed Q16.16 zapisany jako uint32."""
    return int(round(float(value) * Q16_SCALE)) & 0xFFFF_FFFF


def from_q16(raw: int) -> float:
    """uint32 z rejestru -> signed Q16.16 float."""
    raw = int(raw) & 0xFFFF_FFFF
    if raw & 0x8000_0000:
        raw -= 0x1_0000_0000
    return raw / Q16_SCALE


def write_reg(offset: int, value: int) -> None:
    ip.write(int(offset), int(value) & 0xFFFF_FFFF)


def read_reg(offset: int) -> int:
    return int(ip.read(int(offset))) & 0xFFFF_FFFF


def decode_status(status: int) -> dict:
    return {
        "raw": status,
        "busy": bool(status & 0x1),
        "done": bool(status & 0x2),
        "overflow": bool(status & 0x4),
        "core_valid": bool(status & 0x8),
    }


version = read_reg(REG_VERSION)
print(f"VERSION = 0x{version:08X}")
if version != 0x0001_0000:
    print("UWAGA: wersja IP jest inna niz oczekiwana 0x00010000")


## 4. Szybki test AXI-Lite

Sprawdzenie zapisu i odczytu rejestrow. To jeszcze nie uruchamia obliczen.


In [ ]:
readback_tests = [
    (REG_THRESHOLD, to_q16(0.5), "THRESHOLD = 0.5"),
    (REG_T1,        to_q16(10.0), "T1 = 10.0"),
    (REG_A1,        to_q16(1.25), "A1 = 1.25"),
    (REG_T2,        0x1234_5678,  "T2 raw"),
]

ok_all = True
for offset, value, label in readback_tests:
    write_reg(offset, value)
    got = read_reg(offset)
    ok = got == (value & 0xFFFF_FFFF)
    ok_all &= ok
    print(f"{'OK' if ok else 'FAIL':4s} {label:16s} write=0x{value & 0xFFFF_FFFF:08X} read=0x{got:08X}")

print("AXI-Lite readback:", "PASS" if ok_all else "FAIL")


## 5. Uruchamianie IP

`run_event()` wpisuje probki, ustawia tryb, generuje impuls startu, czeka na `STATUS.done` i odczytuje wynik.


In [ ]:
def clear_done() -> None:
    # CONTROL[8] = clear_done. Ten zapis moze tez ustawic mode=0, co nie przeszkadza.
    write_reg(REG_CONTROL, 1 << 8)


def run_event(t1, a1, t2, a2, t3, a3, threshold, mode, timeout_s=0.1):
    if mode not in MODE_NAMES:
        raise ValueError(f"Nieznany mode={mode}")

    clear_done()

    write_reg(REG_THRESHOLD, to_q16(threshold))
    write_reg(REG_T1, to_q16(t1))
    write_reg(REG_A1, to_q16(a1))
    write_reg(REG_T2, to_q16(t2))
    write_reg(REG_A2, to_q16(a2))
    write_reg(REG_T3, to_q16(t3))
    write_reg(REG_A3, to_q16(a3))

    control = ((int(mode) & 0x3) << 1) | 0x1
    write_reg(REG_CONTROL, control)

    deadline = time.perf_counter() + timeout_s
    status = read_reg(REG_STATUS)
    while not (status & 0x2):  # done
        if time.perf_counter() > deadline:
            return {
                "mode": MODE_NAMES[mode],
                "t0_est": np.nan,
                "amax_est": np.nan,
                "timeout": True,
                **decode_status(status),
            }
        status = read_reg(REG_STATUS)

    return {
        "mode": MODE_NAMES[mode],
        "t0_est": from_q16(read_reg(REG_T0_EST)),
        "amax_est": from_q16(read_reg(REG_AMAX_EST)),
        "timeout": False,
        **decode_status(status),
    }


print("run_event() gotowe")


## 6. Referencje zgodne z RTL

Te wzory sprawdzaja aktualny RTL, a nie idealny model fizyczny:

- **LINEAR**: `t1 - a1 * (t2 - t1)/(a2 - a1)`, czyli przeciecie prostej z amplituda 0.
- **EXP**: `tau` z probek 2-3 w dziedzinie `ln(A)`, potem cofniecie probki 1 do `threshold`.
- **LOG**: wierzcholek paraboli dopasowanej do `ln(A)`, czyli estymacja szczytu.


In [ ]:
def ref_linear(t1, a1, t2, a2, threshold=None):
    da21 = a2 - a1
    if da21 == 0:
        return np.nan
    return t1 - a1 * (t2 - t1) / da21


def ref_exp(t1, a1, t2, a2, t3, a3, threshold):
    if min(a1, a2, a3, threshold) <= 0:
        return np.nan
    dln32 = np.log(a3) - np.log(a2)
    if dln32 == 0:
        return np.nan
    tau = (t3 - t2) / dln32
    return t1 - tau * (np.log(a1) - np.log(threshold))


def ref_log(t1, a1, t2, a2, t3, a3):
    if min(a1, a2, a3) <= 0:
        return np.nan
    x = np.array([t1, t2, t3], dtype=float)
    y = np.log(np.array([a1, a2, a3], dtype=float))
    if len(np.unique(x)) != 3:
        return np.nan
    c2, c1, _ = np.polyfit(x, y, 2)
    if c2 >= 0:
        return np.nan
    return -c1 / (2.0 * c2)


def reference_for_mode(row, mode):
    t1, a1, t2, a2, t3, a3, threshold = [row[k] for k in ["t1", "a1", "t2", "a2", "t3", "a3", "threshold"]]
    if mode == MODE_LINEAR:
        return ref_linear(t1, a1, t2, a2, threshold)
    if mode == MODE_EXP:
        return ref_exp(t1, a1, t2, a2, t3, a3, threshold)
    if mode == MODE_LOG:
        return ref_log(t1, a1, t2, a2, t3, a3)
    return np.nan


print("Referencje gotowe")


## 7. Trzy deterministyczne testy

Wyniki oczekiwane:

- LINEAR ok. `9.0`,
- EXP ok. `8.0`,
- LOG ok. `12.0`.


In [ ]:
test_cases = [
    {
        "name": "linear_simple",
        "mode": MODE_LINEAR,
        "t1": 10.0, "a1": 1.0,
        "t2": 12.0, "a2": 3.0,
        "t3": 14.0, "a3": 5.0,
        "threshold": 0.5,
        "expected_about": 9.0,
    },
    {
        "name": "exp_simple",
        "mode": MODE_EXP,
        "t1": 10.0, "a1": float(np.exp(0.5)),
        "t2": 12.0, "a2": float(np.exp(1.0)),
        "t3": 14.0, "a3": float(np.exp(1.5)),
        "threshold": 1.0,
        "expected_about": 8.0,
    },
    {
        "name": "log_peak_simple",
        "mode": MODE_LOG,
        "t1": 10.0, "a1": float(np.exp(3.5)),
        "t2": 12.0, "a2": float(np.exp(4.0)),
        "t3": 14.0, "a3": float(np.exp(3.5)),
        "threshold": 1.0,
        "expected_about": 12.0,
    },
]

rows = []
for case in test_cases:
    result = run_event(case["t1"], case["a1"], case["t2"], case["a2"], case["t3"], case["a3"], case["threshold"], case["mode"])
    ref = reference_for_mode(case, case["mode"])
    rows.append({
        "case": case["name"],
        "mode": result["mode"],
        "fpga_t0": result["t0_est"],
        "python_ref": ref,
        "expected_about": case["expected_about"],
        "error_vs_ref": result["t0_est"] - ref,
        "amax_est": result["amax_est"],
        "done": result["done"],
        "valid": result["core_valid"],
        "overflow": result["overflow"],
        "timeout": result["timeout"],
        "status_raw": f"0x{result['raw']:08X}",
    })

pd.DataFrame(rows)


## 8. Opcjonalny test CSV

Jesli obok notebooka lezy `example_samples.csv`, notebook uruchomi pierwsze zdarzenia z pliku.
Oczekiwane kolumny: `t1,A1,t2,A2,t3,A3`. Kolumna `threshold` jest opcjonalna.


In [ ]:
CSV_FILE = Path("example_samples.csv")
DEFAULT_THRESHOLD = 0.5
MAX_EVENTS = 20

if not CSV_FILE.exists():
    print(f"Brak {CSV_FILE}; pomijam test CSV.")
else:
    data = pd.read_csv(CSV_FILE)
    print(f"Wczytano {len(data)} wierszy z {CSV_FILE}")
    display(data.head())

    required = ["t1", "A1", "t2", "A2", "t3", "A3"]
    missing = [c for c in required if c not in data.columns]
    if missing:
        raise ValueError(f"Brak wymaganych kolumn CSV: {missing}")

    out = []
    for idx, r in data.head(MAX_EVENTS).iterrows():
        row = {
            "t1": float(r["t1"]), "a1": float(r["A1"]),
            "t2": float(r["t2"]), "a2": float(r["A2"]),
            "t3": float(r["t3"]), "a3": float(r["A3"]),
            "threshold": float(r["threshold"]) if "threshold" in data.columns else DEFAULT_THRESHOLD,
        }
        event_id = int(r["event_id"]) if "event_id" in data.columns else int(idx)
        for mode in [MODE_LINEAR, MODE_EXP, MODE_LOG]:
            res = run_event(row["t1"], row["a1"], row["t2"], row["a2"], row["t3"], row["a3"], row["threshold"], mode)
            ref = reference_for_mode(row, mode)
            out.append({
                "event_id": event_id,
                "mode": MODE_NAMES[mode],
                "fpga_t0": res["t0_est"],
                "python_ref": ref,
                "error": res["t0_est"] - ref,
                "amax_est": res["amax_est"],
                "overflow": res["overflow"],
                "valid": res["core_valid"],
                "status": f"0x{res['raw']:08X}",
            })

    results = pd.DataFrame(out)
    display(results)

    ok = results[np.isfinite(results["error"])]
    if len(ok):
        display(ok.groupby("mode")["error"].agg(["count", "mean", "std", "min", "max"]))
        ax = ok.boxplot(column="error", by="mode", grid=True)
        ax.set_title("Blad FPGA - Python reference")
        ax.set_ylabel("error")
        plt.suptitle("")
        plt.show()


## 9. Minimalna procedura software

Sekwencja dla dowolnego kodu sterujacego:

1. wpisz `THRESHOLD`, `T1/A1/T2/A2/T3/A3`,
2. wpisz `CONTROL = (mode << 1) | 1`,
3. czytaj `STATUS`, az `done == 1`,
4. odczytaj `T0_EST` i `AMAX_EST`,
5. opcjonalnie wyczysc `done` przez `CONTROL[8] = 1`.

Adres bazowy w aktualnym Block Design to zwykle `0xA0000000`, ale notebook bierze go z HWH.
